# Phase 5: Campaign Response Prediction

**Objective**: Build classification models to predict customer response to marketing campaigns

**Key Focus Areas**:
- Feature importance and interpretability
- Precision/recall tradeoffs (false positives cost money)
- Business implications over raw accuracy
- Model comparison: Logistic Regression → Random Forest → XGBoost

## 1. Setup and Data Loading

In [16]:
# Core libraries
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# ML libraries
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

# Evaluation metrics
from sklearn.metrics import (
    classification_report, 
    confusion_matrix, 
    roc_auc_score, 
    roc_curve,
    precision_recall_curve,
    average_precision_score,
    precision_score,
    recall_score,
    f1_score
)

# Settings
pd.set_option('display.max_columns', None)
np.random.seed(42)

In [17]:
# Load data with RFM features
df = pd.read_csv('../data/processed/ifood_df_with_rfm.csv')
df.head()

,Income,Kidhome,Teenhome,Recency,MntWines,MntFruits,MntMeatProducts,MntFishProducts,MntSweetProducts,MntGoldProds,NumDealsPurchases,NumWebPurchases,NumCatalogPurchases,NumStorePurchases,NumWebVisitsMonth,AcceptedCmp3,AcceptedCmp4,AcceptedCmp5,AcceptedCmp1,AcceptedCmp2,Complain,Z_CostContact,Z_Revenue,Response,Age,Customer_Days,marital_Divorced,marital_Married,marital_Single,marital_Together,marital_Widow,education_Basic,education_PhD,MntTotal,MntRegularProds,AcceptedCmpOverall,education_Masters,education_Bachelors,TotalChildren,Cluster,Cluster_Name,R,F,M,R_Score,F_Score,M_Score,RFM_Score,RFM_Sum,RFM_Segment,Value_Tier
0,58138.0,0,0,58,635,88,546,172,88,88,3,8,10,4,7,0,0,0,0,0,0,3,11,1,63,2822,0,0,1,0,0,0,0,1529,1441,0,0,1,0,2,Empty Nesters,58,25,1529,3,5,5,355,13,Loyal Customers,High Value
1,46344.0,1,1,38,11,1,6,2,1,6,2,1,1,2,5,0,0,0,0,0,0,3,11,0,66,2272,0,0,1,0,0,0,0,21,15,0,0,1,2,3,Stretched Parents,38,6,21,4,1,1,411,6,Recent Customers,Low Value
2,71613.0,0,0,26,426,49,127,111,21,42,1,8,2,10,4,0,0,0,0,0,0,3,11,0,55,2471,0,0,0,1,0,0,0,734,692,0,0,1,0,2,Empty Nesters,26,21,734,4,4,4,444,12,Champions,High Value
3,26646.0,1,0,26,11,4,20,10,3,5,2,2,0,4,6,0,0,0,0,0,0,3,11,0,36,2298,0,0,0,1,0,0,0,48,43,0,0,1,1,1,Young Families on Budget,26,8,48,4,2,2,422,8,Potential Loyalists,Low Value
4,58293.0,1,0,94,173,43,118,46,27,15,5,5,3,6,5,0,0,0,0,0,0,3,11,0,39,2320,0,1,0,0,0,0,1,407,392,0,0,0,1,1,Young Families on Budget,94,19,407,1,4,3,143,8,At Risk,Low Value


### Dataset Overview

We're working with **2,205 customers** with an overall **campaign response rate of 15.1%**. This represents a class imbalance we'll need to address in our modeling.

**Class Distribution:**
- Non-responders: 1,872 (84.9%)
- Responders: 333 (15.1%)

## 2. Exploratory Analysis: Response Patterns

In [18]:
# Calculate response rates by different segments
response_by_segment = df.groupby('RFM_Segment')['Response'].agg(['sum', 'count', 'mean']).reset_index()
response_by_segment.columns = ['RFM_Segment', 'Responders', 'Total', 'Response_Rate']
response_by_segment = response_by_segment.sort_values('Response_Rate', ascending=False)

response_by_cluster = df.groupby('Cluster_Name')['Response'].agg(['sum', 'count', 'mean']).reset_index()
response_by_cluster.columns = ['Cluster_Name', 'Responders', 'Total', 'Response_Rate']
response_by_cluster = response_by_cluster.sort_values('Response_Rate', ascending=False)

response_by_tier = df.groupby('Value_Tier')['Response'].agg(['sum', 'count', 'mean']).reset_index()
response_by_tier.columns = ['Value_Tier', 'Responders', 'Total', 'Response_Rate']
tier_order = ['High Value', 'Medium Value', 'Low Value', 'Very Low Value']
response_by_tier['Value_Tier'] = pd.Categorical(response_by_tier['Value_Tier'], categories=tier_order, ordered=True)
response_by_tier = response_by_tier.sort_values('Value_Tier')

response_by_past = df.groupby('AcceptedCmpOverall')['Response'].agg(['sum', 'count', 'mean']).reset_index()
response_by_past.columns = ['AcceptedCmpOverall', 'Responders', 'Total', 'Response_Rate']

In [19]:

# Create subplots
fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=('Response Rate by RFM Segment', 'Response Rate by Customer Cluster',
                    'Response Rate by Value Tier', 'Response Rate by Past Campaign Acceptance'),
    specs=[[{'type': 'bar'}, {'type': 'bar'}],
           [{'type': 'bar'}, {'type': 'scatter'}]]
)

# RFM Segment
fig.add_trace(
    go.Bar(x=response_by_segment['RFM_Segment'], y=response_by_segment['Response_Rate'],
           marker_color='steelblue', name='RFM Segment', showlegend=False,
           hovertemplate='<b>%{x}</b><br>Response Rate: %{y:.1%}<extra></extra>'),
    row=1, col=1
)

# Cluster
fig.add_trace(
    go.Bar(x=response_by_cluster['Cluster_Name'], y=response_by_cluster['Response_Rate'],
           marker_color='coral', name='Cluster', showlegend=False,
           hovertemplate='<b>%{x}</b><br>Response Rate: %{y:.1%}<extra></extra>'),
    row=1, col=2
)

# Value Tier
fig.add_trace(
    go.Bar(x=response_by_tier['Value_Tier'], y=response_by_tier['Response_Rate'],
           marker_color='seagreen', name='Value Tier', showlegend=False,
           hovertemplate='<b>%{x}</b><br>Response Rate: %{y:.1%}<extra></extra>'),
    row=2, col=1
)

# Past Campaign Acceptance
fig.add_trace(
    go.Scatter(x=response_by_past['AcceptedCmpOverall'], y=response_by_past['Response_Rate'],
              mode='lines+markers', marker=dict(size=10, color='orange'),
              line=dict(width=3, color='orange'), name='Past Campaigns', showlegend=False,
              hovertemplate='<b>%{x} campaigns</b><br>Response Rate: %{y:.1%}<extra></extra>'),
    row=2, col=2
)

# Add horizontal baseline (overall avg)
overall_avg = df['Response'].mean()
for i in range(1, 3):
    for j in range(1, 3):
        fig.add_hline(y=overall_avg, line_dash='dash', line_color='red', opacity=0.5,
                     annotation_text=f'Overall Avg: {overall_avg:.1%}' if i==1 and j==1 else '',
                     row=i, col=j)

fig.update_yaxes(tickformat='.0%', title_text='Response Rate', row=1, col=1)
fig.update_yaxes(tickformat='.0%', title_text='Response Rate', row=1, col=2)
fig.update_yaxes(tickformat='.0%', title_text='Response Rate', row=2, col=1)
fig.update_yaxes(tickformat='.0%', title_text='Response Rate', row=2, col=2)

fig.update_xaxes(title_text='# Past Campaigns Accepted', row=2, col=2)

fig.update_layout(height=700, showlegend=False, title_text='Campaign Response Analysis by Customer Segments')
fig.show()

### Key Insights from Response Analysis

**1. RFM Segments:**
- **Champions** have the highest response rate at **29.8%** (nearly 2x the average)
- **Potential Loyalists** follow at **24.3%**
- **Lost** and **Hibernating** segments show minimal response (<5%)

**2. Customer Clusters:**
- **Empty Nesters** respond at **31.2%** - significantly above average
- **Young Families on Budget** and **Stretched Parents** show low engagement (<13%)

**3. Value Tiers:**
- Clear linear relationship: **High Value** customers respond at **30.8%**
- **Very Low Value** customers barely respond (3.1%)

**4. Past Campaign Behavior:**
- **Strongest predictor**: Past campaign acceptance shows dramatic correlation
- Customers who accepted 4 campaigns respond at **90.9%** to the next one
- This suggests strong behavioral persistence

## 3. Feature Preparation

In [20]:
# Define feature sets
exclude_cols = ['Response', 'ID']

# Select numerical features
numerical_features = df.select_dtypes(include=[np.number]).columns.tolist()
numerical_features = [f for f in numerical_features if f not in exclude_cols]

# Prepare feature matrix
X = df[numerical_features].copy()

# Add categorical features as dummies
categorical_features = ['RFM_Segment', 'Value_Tier', 'Cluster_Name']
for col in categorical_features:
    dummies = pd.get_dummies(df[col], prefix=col, drop_first=True)
    X = pd.concat([X, dummies], axis=1)

# Target variable
y = df['Response']

print(f"✓ Feature matrix: {X.shape[0]:,} customers × {X.shape[1]} features")
print(f"✓ Target distribution: {(y==0).sum():,} non-responders, {(y==1).sum():,} responders ({y.mean():.1%})")

✓ Feature matrix: 2,205 customers × 61 features
✓ Target distribution: 1,872 non-responders, 333 responders (15.1%)


In [21]:
# Train-test split with stratification
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Scale features
scaler = StandardScaler()
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train), columns=X_train.columns, index=X_train.index)
X_test_scaled = pd.DataFrame(scaler.transform(X_test), columns=X_test.columns, index=X_test.index)

print(f"✓ Training set: {X_train.shape[0]:,} customers ({y_train.mean():.1%} response rate)")
print(f"✓ Test set: {X_test.shape[0]:,} customers ({y_test.mean():.1%} response rate)")

✓ Training set: 1,764 customers (15.1% response rate)
✓ Test set: 441 customers (15.2% response rate)


## 4. Model Training

We'll train three models with increasing complexity:
1. **Logistic Regression** - Baseline, highly interpretable
2. **Random Forest** - Captures non-linear relationships
3. **XGBoost** - State-of-the-art gradient boosting

All models use `class_weight='balanced'` to handle the 85/15 class imbalance.

In [22]:
# Train all three models
print("Training models...")

# 1. Logistic Regression
lr_model = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)
lr_model.fit(X_train_scaled, y_train)
y_pred_lr = lr_model.predict(X_test_scaled)
y_pred_proba_lr = lr_model.predict_proba(X_test_scaled)[:, 1]

# 2. Random Forest
rf_model = RandomForestClassifier(
    n_estimators=200, max_depth=10, min_samples_split=20, min_samples_leaf=10,
    class_weight='balanced', random_state=42, n_jobs=-1
)
rf_model.fit(X_train, y_train)
y_pred_rf = rf_model.predict(X_test)
y_pred_proba_rf = rf_model.predict_proba(X_test)[:, 1]

# 3. XGBoost
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()
xgb_model = XGBClassifier(
    n_estimators=200, max_depth=6, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8,
    scale_pos_weight=scale_pos_weight, random_state=42, eval_metric='logloss'
)
xgb_model.fit(X_train, y_train)
y_pred_xgb = xgb_model.predict(X_test)
y_pred_proba_xgb = xgb_model.predict_proba(X_test)[:, 1]

print("✓ All models trained successfully")

✓ All models trained successfully


## 5. Model Performance Comparison

In [23]:
# Calculate metrics for all models
cm_lr = confusion_matrix(y_test, y_pred_lr)
cm_rf = confusion_matrix(y_test, y_pred_rf)
cm_xgb = confusion_matrix(y_test, y_pred_xgb)

comparison_df = pd.DataFrame({
    'Model': ['Logistic Regression', 'Random Forest', 'XGBoost'],
    'ROC-AUC': [
        roc_auc_score(y_test, y_pred_proba_lr),
        roc_auc_score(y_test, y_pred_proba_rf),
        roc_auc_score(y_test, y_pred_proba_xgb)
    ],
    'Precision': [
        precision_score(y_test, y_pred_lr),
        precision_score(y_test, y_pred_rf),
        precision_score(y_test, y_pred_xgb)
    ],
    'Recall': [
        recall_score(y_test, y_pred_lr),
        recall_score(y_test, y_pred_rf),
        recall_score(y_test, y_pred_xgb)
    ],
    'F1-Score': [
        f1_score(y_test, y_pred_lr),
        f1_score(y_test, y_pred_rf),
        f1_score(y_test, y_pred_xgb)
    ],
    'False Positives': [cm_lr[0,1], cm_rf[0,1], cm_xgb[0,1]],
    'False Negatives': [cm_lr[1,0], cm_rf[1,0], cm_xgb[1,0]]
})

In [31]:
# Create comparison visualization
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=('Performance Metrics Comparison', 'False Positives vs False Negatives'),
    specs=[[{'type': 'bar'}, {'type': 'bar'}]]
)

# Metrics comparison
metrics = ['ROC-AUC', 'Precision', 'Recall', 'F1-Score']
colors = ['#3498db', '#2ecc71', '#e74c3c']
for i, model in enumerate(comparison_df['Model']):
    fig.add_trace(
        go.Bar(name=model, x=metrics, y=comparison_df.loc[i, metrics].values,
               marker_color=colors[i], showlegend=True),
        row=1, col=1
    )

# Error comparison
fig.add_trace(
    go.Bar(name='False Positives (Cost $)', x=comparison_df['Model'],
           y=comparison_df['False Positives'], marker_color='#e74c3c', showlegend=False),
    row=1, col=2
)
fig.add_trace(
    go.Bar(name='False Negatives (Lost $)', x=comparison_df['Model'],
           y=comparison_df['False Negatives'], marker_color='#f39c12', showlegend=False),
    row=1, col=2
)

fig.update_layout(height=400, barmode='group', title_text='Model Performance Overview')
fig.update_yaxes(title_text='Score', range=[0, 1], row=1, col=1)
fig.update_yaxes(title_text='Count', row=1, col=2)
fig.show()

In [30]:
comparison_df

,Model,ROC-AUC,Precision,Recall,F1-Score,False Positives,False Negatives
0,Logistic Regression,0.908213,0.486957,0.835821,0.615385,59,11
1,Random Forest,0.893128,0.500000,0.611940,0.550336,41,26
2,XGBoost,0.904142,0.583333,0.522388,0.551181,25,32


### Performance Analysis

**Logistic Regression:**
- **Best ROC-AUC** (0.908) and **highest recall** (0.84)
- Catches 84% of all responders but has 59 false positives
- Most interpretable - clear coefficient meanings

**Random Forest:**
- Balanced performance across metrics
- **Fewest false positives** (41) but also lower recall (0.61)
- More conservative in predictions

**XGBoost:**
- **Best precision** (0.58) and **fewest false positives** (25)
- Most conservative - only targets high-confidence responders
- Lowest recall (0.52) means missing potential responders

**The Tradeoff:** High recall (LR) catches more customers but costs more in false positives. High precision (XGB) minimizes waste but misses opportunities.

## 6. ROC and Precision-Recall Curves

In [25]:
# ROC and PR curves
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=('ROC Curves', 'Precision-Recall Curves')
)

models = [
    ('Logistic Regression', y_pred_proba_lr, '#3498db'),
    ('Random Forest', y_pred_proba_rf, '#2ecc71'),
    ('XGBoost', y_pred_proba_xgb, '#e74c3c')
]

# ROC Curves
for name, proba, color in models:
    fpr, tpr, _ = roc_curve(y_test, proba)
    auc = roc_auc_score(y_test, proba)
    fig.add_trace(
        go.Scatter(x=fpr, y=tpr, name=f'{name} (AUC={auc:.3f})',
                   line=dict(color=color, width=3),
                   hovertemplate='FPR: %{x:.3f}<br>TPR: %{y:.3f}<extra></extra>'),
        row=1, col=1
    )

# Random baseline
fig.add_trace(
    go.Scatter(x=[0, 1], y=[0, 1], name='Random',
               line=dict(color='gray', width=2, dash='dash'), showlegend=False),
    row=1, col=1
)

# Precision-Recall Curves
baseline_precision = y_test.mean()
for name, proba, color in models:
    precision, recall, _ = precision_recall_curve(y_test, proba)
    avg_precision = average_precision_score(y_test, proba)
    fig.add_trace(
        go.Scatter(x=recall, y=precision, name=f'{name} (AP={avg_precision:.3f})',
                   line=dict(color=color, width=3),
                   hovertemplate='Recall: %{x:.3f}<br>Precision: %{y:.3f}<extra></extra>',
                   showlegend=False),
        row=1, col=2
    )

fig.update_xaxes(title_text='False Positive Rate', row=1, col=1)
fig.update_yaxes(title_text='True Positive Rate', row=1, col=1)
fig.update_xaxes(title_text='Recall', row=1, col=2)
fig.update_yaxes(title_text='Precision', row=1, col=2)
fig.update_layout(height=450, title_text='Model Discrimination Analysis')
fig.show()

## 7. Feature Importance Analysis

In [26]:
# Extract feature importance from all models
feature_importance_lr = pd.DataFrame({
    'feature': X_train.columns,
    'importance': np.abs(lr_model.coef_[0]),
    'direction': lr_model.coef_[0]
}).sort_values('importance', ascending=False).head(15)

feature_importance_rf = pd.DataFrame({
    'feature': X_train.columns,
    'importance': rf_model.feature_importances_
}).sort_values('importance', ascending=False).head(15)

feature_importance_xgb = pd.DataFrame({
    'feature': X_train.columns,
    'importance': xgb_model.feature_importances_
}).sort_values('importance', ascending=False).head(15)

# Create comparison plot
fig = make_subplots(
    rows=1, cols=3,
    subplot_titles=('Logistic Regression<br>(Coefficient Magnitude)', 
                    'Random Forest<br>(Gini Importance)',
                    'XGBoost<br>(Gain)'),
    specs=[[{'type': 'bar'}, {'type': 'bar'}, {'type': 'bar'}]]
)

# Logistic Regression - color by direction
colors_lr = ['green' if x > 0 else 'red' for x in feature_importance_lr['direction']]
fig.add_trace(
    go.Bar(y=feature_importance_lr['feature'], x=feature_importance_lr['importance'],
           orientation='h', marker_color=colors_lr, showlegend=False,
           hovertemplate='%{y}<br>Coefficient: %{x:.3f}<extra></extra>'),
    row=1, col=1
)

# Random Forest
fig.add_trace(
    go.Bar(y=feature_importance_rf['feature'], x=feature_importance_rf['importance'],
           orientation='h', marker_color='#2ecc71', showlegend=False,
           hovertemplate='%{y}<br>Importance: %{x:.3f}<extra></extra>'),
    row=1, col=2
)

# XGBoost
fig.add_trace(
    go.Bar(y=feature_importance_xgb['feature'], x=feature_importance_xgb['importance'],
           orientation='h', marker_color='#e74c3c', showlegend=False,
           hovertemplate='%{y}<br>Importance: %{x:.3f}<extra></extra>'),
    row=1, col=3
)

fig.update_xaxes(title_text='Importance', row=1, col=1)
fig.update_xaxes(title_text='Importance', row=1, col=2)
fig.update_xaxes(title_text='Importance', row=1, col=3)
fig.update_layout(height=500, title_text='Top 15 Features by Model')
fig.show()

### Feature Importance Insights

**Consensus Features (Important Across All Models):**
1. **AcceptedCmpOverall** - Past campaign behavior is the strongest predictor
2. **Customer_Days** - Customer tenure/recency
3. **M_Score / MntTotal** - Monetary value
4. **R_Score / Recency** - How recently they purchased

**Key Insights from Logistic Regression Coefficients:**
- **Green (Positive):** More of this = Higher response probability
  - Past campaign acceptance
  - Customer tenure
  - Monetary value
  - Catalog purchases
  
- **Red (Negative):** More of this = Lower response probability
  - Store purchases (online customers respond better)
  - Recency (recent buyers already engaged)
  - Deal purchases (price-sensitive customers)

**Business Translation:**
The best campaign targets are **established, high-spending customers who have engaged with past campaigns and prefer catalog/web channels over stores**.

## 8. Profit Optimization: Finding the Right Threshold

**Business Context:**
- **Cost per contact**: $10 (campaign materials, labor, etc.)
- **Revenue per conversion**: $50 (average order value)
- **Goal**: Maximize profit = (Conversions × $50) - (Contacts × $10)

Default threshold (0.5) isn't optimized for profit. Let's find the threshold that maximizes ROI.

In [27]:
def calculate_profit(y_true, y_pred_proba, threshold, cost_per_contact=10, revenue_per_response=50):
    """Calculate profit metrics at a given threshold"""
    y_pred = (y_pred_proba >= threshold).astype(int)
    tp = ((y_pred == 1) & (y_true == 1)).sum()
    fp = ((y_pred == 1) & (y_true == 0)).sum()
    
    total_contacts = tp + fp
    total_cost = total_contacts * cost_per_contact
    total_revenue = tp * revenue_per_response
    profit = total_revenue - total_cost
    
    return {
        'profit': profit,
        'revenue': total_revenue,
        'cost': total_cost,
        'contacts': total_contacts,
        'conversions': tp,
        'precision': tp / total_contacts if total_contacts > 0 else 0,
        'recall': tp / y_true.sum() if y_true.sum() > 0 else 0
    }

# Calculate profit across thresholds
thresholds = np.arange(0.1, 0.9, 0.05)
results_lr = [calculate_profit(y_test, y_pred_proba_lr, t) for t in thresholds]
results_rf = [calculate_profit(y_test, y_pred_proba_rf, t) for t in thresholds]
results_xgb = [calculate_profit(y_test, y_pred_proba_xgb, t) for t in thresholds]

df_results_lr = pd.DataFrame(results_lr, index=thresholds)
df_results_rf = pd.DataFrame(results_rf, index=thresholds)
df_results_xgb = pd.DataFrame(results_xgb, index=thresholds)

optimal_threshold_lr = df_results_lr['profit'].idxmax()
optimal_threshold_rf = df_results_rf['profit'].idxmax()
optimal_threshold_xgb = df_results_xgb['profit'].idxmax()

In [28]:
# Visualize threshold optimization
fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=('Profit vs Threshold', 'Precision-Recall Tradeoff',
                    'Campaign Size vs Threshold', 'Conversion Rate vs Threshold')
)

models_data = [
    ('Logistic Regression', df_results_lr, optimal_threshold_lr, '#3498db'),
    ('Random Forest', df_results_rf, optimal_threshold_rf, '#2ecc71'),
    ('XGBoost', df_results_xgb, optimal_threshold_xgb, '#e74c3c')
]

for name, df_results, opt_thresh, color in models_data:
    # Profit
    fig.add_trace(
        go.Scatter(x=thresholds, y=df_results['profit'], name=name,
                   line=dict(color=color, width=2), mode='lines+markers',
                   hovertemplate='%{y:$,.0f}<extra></extra>'),
        row=1, col=1
    )
    fig.add_vline(x=opt_thresh, line_dash='dash', line_color=color, opacity=0.3, row=1, col=1)
    
    # Precision vs Recall
    fig.add_trace(
        go.Scatter(x=df_results['recall'], y=df_results['precision'], name=name,
                   line=dict(color=color, width=2), showlegend=False,
                   hovertemplate='Recall: %{x:.1%}<br>Precision: %{y:.1%}<extra></extra>'),
        row=1, col=2
    )
    
    # Contacts
    fig.add_trace(
        go.Scatter(x=thresholds, y=df_results['contacts'], name=name,
                   line=dict(color=color, width=2), showlegend=False,
                   hovertemplate='%{y} customers<extra></extra>'),
        row=2, col=1
    )
    fig.add_vline(x=opt_thresh, line_dash='dash', line_color=color, opacity=0.3, row=2, col=1)
    
    # Conversion rate
    fig.add_trace(
        go.Scatter(x=thresholds, y=df_results['precision']*100, name=name,
                   line=dict(color=color, width=2), showlegend=False,
                   hovertemplate='%{y:.1f}%<extra></extra>'),
        row=2, col=2
    )
    fig.add_vline(x=opt_thresh, line_dash='dash', line_color=color, opacity=0.3, row=2, col=2)

fig.update_xaxes(title_text='Threshold', row=1, col=1)
fig.update_xaxes(title_text='Recall', row=1, col=2)
fig.update_xaxes(title_text='Threshold', row=2, col=1)
fig.update_xaxes(title_text='Threshold', row=2, col=2)

fig.update_yaxes(title_text='Profit ($)', row=1, col=1)
fig.update_yaxes(title_text='Precision', row=1, col=2)
fig.update_yaxes(title_text='Customers Contacted', row=2, col=1)
fig.update_yaxes(title_text='Conversion Rate (%)', row=2, col=2)

fig.update_layout(height=700, title_text='Threshold Optimization Analysis')
fig.show()

In [29]:
# Optimal thresholds comparison
best_results = pd.DataFrame({
    'Model': ['Logistic Regression', 'Random Forest', 'XGBoost'],
    'Optimal Threshold': [optimal_threshold_lr, optimal_threshold_rf, optimal_threshold_xgb],
    'Expected Profit': [
        df_results_lr.loc[optimal_threshold_lr, 'profit'],
        df_results_rf.loc[optimal_threshold_rf, 'profit'],
        df_results_xgb.loc[optimal_threshold_xgb, 'profit']
    ],
    'Customers to Contact': [
        int(df_results_lr.loc[optimal_threshold_lr, 'contacts']),
        int(df_results_rf.loc[optimal_threshold_rf, 'contacts']),
        int(df_results_xgb.loc[optimal_threshold_xgb, 'contacts'])
    ],
    'Expected Conversions': [
        int(df_results_lr.loc[optimal_threshold_lr, 'conversions']),
        int(df_results_rf.loc[optimal_threshold_rf, 'conversions']),
        int(df_results_xgb.loc[optimal_threshold_xgb, 'conversions'])
    ],
    'Conversion Rate': [
        f"{df_results_lr.loc[optimal_threshold_lr, 'precision']*100:.1f}%",
        f"{df_results_rf.loc[optimal_threshold_rf, 'precision']*100:.1f}%",
        f"{df_results_xgb.loc[optimal_threshold_xgb, 'precision']*100:.1f}%"
    ]
})

best_results

,Model,Optimal Threshold,Expected Profit,Customers to Contact,Expected Conversions,Conversion Rate
0,Logistic Regression,0.45,1680,127,59,46.5%
1,Random Forest,0.35,1590,146,61,41.8%
2,XGBoost,0.10,1550,145,60,41.4%


## 9. Final Recommendations

### Recommended Model: **Logistic Regression at 0.45 threshold**

**Why Logistic Regression?**
1. **Highest profit**: $1,680 (5.6% better than alternatives)
2. **Best conversion rate**: 46.5% (3-in-1 customers respond)
3. **Interpretability**: Clear coefficient meanings for stakeholder buy-in
4. **Simplicity**: Fast predictions for real-time scoring
5. **Best ROC-AUC**: Superior discrimination ability (0.908)

**Campaign Strategy:**
- **Target**: 127 customers (29% of test set)
- **Expected**: 59 conversions (46.5% conversion rate)
- **Profit**: $1,680 per campaign cycle
- **ROI**: 132% (every $1 spent returns $2.32)

### Key Success Factors

**1. Prioritize These Segments:**
- **Champions** (29.8% response rate)
- **Potential Loyalists** (24.3%)
- **Empty Nesters** cluster (31.2%)
- **High Value** tier (30.8%)

**2. Focus on These Features:**
- **Past campaign acceptance** (strongest predictor by far)
- **Customer tenure** (longer = better)
- **Monetary value** (high spenders respond more)
- **Channel preference** (catalog/web > store)

**3. Avoid Targeting:**
- Lost/Hibernating segments (<5% response)
- Very Low Value tier (3.1% response)
- Customers with 0 past campaign acceptance (<10% response)

### Implementation Plan

**Phase 1: Pilot Test (Recommended)**
1. Run A/B test: Model predictions vs current targeting
2. Split test population 50/50
3. Measure lift in conversion rate and profit
4. Duration: 2 campaign cycles

**Phase 2: Full Deployment**
1. Deploy model with 0.45 threshold
2. Monitor weekly for:
   - Conversion rate (target: >40%)
   - False positive rate (target: <15%)
   - Profit per campaign (target: >$1,500)
3. Retrain quarterly with new data

**Phase 3: Optimization**
1. Develop segment-specific models
2. Test personalized messaging by segment
3. Implement dynamic threshold adjustment
4. Explore uplift modeling (incremental impact)

### Risk Mitigation

**Risks:**
- Model drift over time (customer behavior changes)
- Overemphasis on past responders (limits growth)
- False negatives (missing potential high-value customers)

**Mitigations:**
- Monthly performance monitoring dashboard
- Quarterly model retraining
- Reserve 10% of campaign for exploration (random sampling)
- Track long-term customer value, not just immediate response

### Expected Business Impact

**Current Campaign (Baseline: 15% response, contact all 2,205 customers):**
- Contacts: 2,205
- Conversions: ~333 (15%)
- Revenue: $16,650
- Cost: $22,050
- **Profit: -$5,400 (LOSS)**

**Optimized Campaign (Model: 46.5% response, contact 127 customers):**
- Contacts: 127
- Conversions: ~59 (46.5%)
- Revenue: $2,950
- Cost: $1,270
- **Profit: +$1,680 (GAIN)**

**Net Improvement: $7,080 per campaign** 🎯